# LLM Common — Eval Mechanics Playground

**~90 minutes · Narayanan Palani · Jupyter notebook **

Today's session builds the four eval patterns you'll port to your capstone in the next phase. We do it on a small, non-capstone dataset first — 6 social media platform summaries — so the mechanics are the ONLY thing you're thinking about.

In the upcoming phase you'll build a 20-question golden set for YOUR capstone corpus and run these same patterns against your own `/ask` endpoint. That's the milestone (M1). Today is the mental model.

**The four patterns we build:**
1. Rubric-based LLM-as-judge (Section 4) — score answers 1-4 on 3 dimensions
2. Pairwise comparison (Section 5) — 'which is better, A or B?'
3. Position bias — see it happen, then mitigate (Section 6)
4. Critic-Creator loop (Section 7) — systematic prompt improvement

**Cost per full run:** $0.00 (Local execution using Ollama: `gemma2:2b` as judge and `gpt-oss:20b` as candidate).

---

## Section 1 — Setup + corpus

6 short social media platform profiles. Small enough to inline. Diverse enough that questions can be easy, tricky, or edge cases.

This corpus stays with us all session.

### Cell 1 Explanation

- **Objective**: Initialize the OpenAI client pointing to a local Ollama server instance and configure flipped judge/candidate model definitions.
- **Purpose**: Establish local LLM connectivity using `gemma2:2b` as the evaluation judge model and `gpt-oss:20b` as the heavy reasoning candidate model.
- **Why we run the code**: To verify system configuration and API parameters before running test suites, ensuring all local API calls route correctly to Ollama without cloud dependencies.
- **What is industry standard**: Decoupling API configuration variables (base URL, model identifiers, keys) from core logic. Note that while production systems typically use a stronger judge model, testing with a lightweight judge (`gemma2:2b`) evaluating a heavier candidate (`gpt-oss:20b`) allows evaluating judge sensitivity and cost/performance trade-offs.

In [1]:
import os
import json
import random
from openai import OpenAI
from pydantic import BaseModel, Field

# Connect to local Ollama API instance via OpenAI client compatibility
client = OpenAI(
    base_url=os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
    api_key="ollama"  # Dummy key required by standard client initialization
)

# Local Ollama Models configuration (Flipped Role Allocation)
# gemma2:2b serves as the lightweight evaluation judge model
# gpt-oss:20b serves as the heavy candidate generation model
JUDGE_MODEL     = "gemma2:2b"
CANDIDATE_MODEL = "gpt-oss:20b"

print(f"Judge model:     {JUDGE_MODEL}")
print(f"Candidate model: {CANDIDATE_MODEL}")
print("Setup ok.")

Judge model:     gemma2:2b
Candidate model: gpt-oss:20b
Setup ok.


---
### Output Analysis — Cell 1

- **What is expected from output**: Printed confirmation showing `gemma2:2b` assigned as `Judge model`, `gpt-oss:20b` as `Candidate model`, and `Setup ok.`
- **How to understand the Output**: Confirms that python modules are correctly loaded and target model aliases are stored in memory with roles inverted.
- **What could potentially go wrong / error handling**: 
  - `ConnectionRefusedError` or `APIConnectionError`: Occurs if Ollama isn't running locally. Fix by starting Ollama daemon (`ollama serve`).
  - `NotFoundError`: Occurs if specified models are not pulled yet. Fix by running `ollama run gemma2:2b` and `ollama run gpt-oss:20b` in terminal.

### Cell 2 Explanation

- **Objective**: Define and print metadata for the evaluation benchmark corpus.
- **Purpose**: Supply a constant in-memory dataset of 6 social media platform summaries used as grounding contexts across all eval patterns.
- **Why we run the code**: Establishes the static source truth against which candidate responses will be generated and judged.
- **What is industry standard**: Storing reference benchmark data in structured JSON format or relational tables with fixed IDs, short descriptions, and calculated character/token counts.

In [2]:
# Corpus data source
PLATFORMS = [
    {"id": "instagram", "title": "Instagram", "details": (
        "Instagram is a photo and short-video sharing social network owned by Meta. "
        "It features Stories, Reels, and direct messaging, focusing heavily on visual branding "
        "and influencer marketing. Users connect via feeds, discovery channels, and visual hashtag curation."
    )},
    {"id": "linkedin", "title": "LinkedIn", "details": (
        "LinkedIn is a professional networking platform operated by Microsoft. "
        "It focuses on career development, job postings, corporate recruiting, and B2B content marketing. "
        "Users build professional credentials and interact via long-form posts, articles, and network messaging."
    )},
    {"id": "tiktok", "title": "TikTok", "details": (
        "TikTok is a short-form video hosting service owned by ByteDance. "
        "It leverages a hyper-personalized recommendation algorithm ('For You Page') to surface dynamic visual content. "
        "The platform emphasizes viral trends, audio soundbites, and creative video editing tools."
    )},
    {"id": "youtube", "title": "YouTube", "details": (
        "YouTube is a video sharing and streaming platform owned by Google. "
        "It supports long-form video content, live streams, and short videos called Shorts. "
        "Content creators monetize through channel memberships, ad revenue, and affiliate marketing integrations."
    )},
    {"id": "x_twitter", "title": "X (formerly Twitter)", "details": (
        "X is a microblogging and real-time social networking platform. "
        "It emphasizes short text posts, breaking news updates, and live public conversations. "
        "Users interact through re-posts, direct quotes, hashtags, and subscription-based premium features."
    )},
    {"id": "pinterest", "title": "Pinterest", "details": (
        "Pinterest is a visual discovery and moodboard platform designed for saving ideas. "
        "Users create curated digital boards containing images and product links called 'Pins'. "
        "It heavily drives e-commerce discovery across lifestyle, design, recipe, and DIY content categories."
    )},
]

print(f"Loaded {len(PLATFORMS)} platforms.\n")
for p in PLATFORMS:
    print(f"  {p['id']:15s}  {p['title']:20s}  {len(p['details'])} chars")

Loaded 6 platforms.

  instagram        Instagram             260 chars
  linkedin         LinkedIn              270 chars
  tiktok           TikTok                265 chars
  youtube          YouTube               254 chars
  x_twitter        X (formerly Twitter)  247 chars
  pinterest        Pinterest             269 chars


---
### Output Analysis — Cell 2

- **What is expected from output**: A listed summary of 6 dataset records showing unique ID, full title, and details length in characters (~310-345 characters per record).
- **How to understand the Output**: Confirms that corpus structures were defined cleanly without syntax errors or missing data keys.
- **What could potentially go wrong / error handling**: `KeyError` if a dictionary missing `title` or `details` keys is processed. Mitigate by using `.get()` with defaults when inspecting arbitrary datasets.

### Cell 3 Explanation

- **Objective**: Define a golden test question set with metadata classifications.
- **Purpose**: Set up evaluation questions varying across difficulties (easy, medium, edge/out-of-scope, multi-hop inference) along with ideal ground-truth reference answers.
- **Why we run the code**: To create a comprehensive eval benchmark that tests models on direct facts, cross-document reasoning, and safety/out-of-bound handling.
- **What is industry standard**: Golden sets must contain categorized edge-cases (e.g., out-of-scope questions to check refusal behavior) alongside simple factual queries.

In [3]:
# Golden Questions benchmark array
QUESTIONS = [
    {"id": "q1", "question": "Which parent company owns LinkedIn?",
     "ideal": "Microsoft", "difficulty": "easy"},
    {"id": "q2", "question": "What is the main recommendation feed on TikTok called?",
     "ideal": "For You Page", "difficulty": "easy"},
    {"id": "q3", "question": "How do content monetization methods differ between YouTube and Pinterest?",
     "ideal": "YouTube relies on channel memberships, ads, and affiliate links; Pinterest focuses on e-commerce discovery and shopping Pins.",
     "difficulty": "medium"},
    {"id": "q4", "question": "Compare how engagement formats differ between X (Twitter) and Instagram.",
     "ideal": "X focuses on short text posts, re-posts, and news updates; Instagram focuses on photos, Reels, Stories, and visual hashtags.",
     "difficulty": "medium"},
    {"id": "q5", "question": "What is the monthly active user count of Snapchat?",
     "ideal": "Not covered — Snapchat is not in the corpus.",
     "difficulty": "edge (out-of-scope)"},
    {"id": "q6", "question": "Which social media platforms in the corpus emphasize visual moodboards and product links?",
     "ideal": "Pinterest.",
     "difficulty": "inference"},
]

print(f"{len(QUESTIONS)} questions loaded.\n")
for q in QUESTIONS:
    print(f"  {q['id']}  ({q['difficulty']:22s}) {q['question']}")

6 questions loaded.

  q1  (easy                  ) Which parent company owns LinkedIn?
  q2  (easy                  ) What is the main recommendation feed on TikTok called?
  q3  (medium                ) How do content monetization methods differ between YouTube and Pinterest?
  q4  (medium                ) Compare how engagement formats differ between X (Twitter) and Instagram.
  q5  (edge (out-of-scope)   ) What is the monthly active user count of Snapchat?
  q6  (inference             ) Which social media platforms in the corpus emphasize visual moodboards and product links?


---
### Output Analysis — Cell 3

- **What is expected from output**: A printed list of 6 benchmark questions alongside their difficulty labels.
- **How to understand the Output**: Validates that all test queries are registered cleanly in memory.
- **What could potentially go wrong / error handling**: Ensure every query dictionary contains matching schema keys (`id`, `question`, `ideal`, `difficulty`) to prevent downstream code failures.

---

## Section 2 — Generate naive answers

Stuff the whole corpus into the prompt (naive — no retrieval). Get an answer to each question. These become our **candidates to grade**.

### Cell 4 Explanation

- **Objective**: Generate initial candidate answers using the primary heavy reasoning candidate model (`gpt-oss:20b`).
- **Purpose**: Execute naive full-context prompt stuffing to obtain responses for each benchmark question.
- **Why we run the code**: Produces high-quality candidate outputs from `gpt-oss:20b` needed for rubric and pairwise evaluation in subsequent sections.
- **What is industry standard**: Using deterministic generation parameters (`temperature=0.0`) when producing standard benchmark baseline candidates.

In [4]:
def answer_naively(question: str, corpus: list, temperature: float = 0.0) -> str:
    """Stuff the whole corpus into the prompt. Ask the question using candidate model (gpt-oss:20b)."""
    context = "\n\n".join(f"=== {p['title']} ===\n{p['details']}" for p in corpus)
    resp = client.chat.completions.create(
        model=CANDIDATE_MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content":
                "You are a helpful assistant. Answer using ONLY the provided "
                "social media profiles. If the answer isn't in the context, say so."},
            {"role": "user", "content":
                f"Platform profiles:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return resp.choices[0].message.content

# Generate one answer per question
candidates = []
for q in QUESTIONS:
    ans = answer_naively(q["question"], PLATFORMS)
    candidates.append({"question_id": q["id"], "answer": ans})
    print(f"── {q['id']} ({q['difficulty']}) ──")
    print(f"   Q: {q['question']}")
    print(f"   A: {ans}\n")

── q1 (easy) ──
   Q: Which parent company owns LinkedIn?
   A: LinkedIn is operated by **Microsoft**.

── q2 (easy) ──
   Q: What is the main recommendation feed on TikTok called?
   A: The main recommendation feed on TikTok is called the **“For You Page”**.

── q3 (medium) ──
   Q: How do content monetization methods differ between YouTube and Pinterest?
   A: YouTube’s description says that creators can **monetize directly** through:

- **Ad revenue** (ads shown on their videos)  
- **Channel memberships** (fans pay a recurring fee for perks)  
- **Affiliate marketing integrations** (promoting products and earning commissions)

Pinterest’s description, on the other hand, does **not mention any direct monetization options** for creators. Instead, it focuses on driving **e‑commerce discovery**—users save and share product‑related pins that link to shopping sites, but creators themselves don’t receive ad revenue or membership fees from Pinterest.

── q4 (medium) ──
   Q: Compare how en

---
### Output Analysis — Cell 4

- **What is expected from output**: Six printed pairs of questions and generated answers produced by `gpt-oss:20b`.
- **How to understand the Output**: Evaluates whether `gpt-oss:20b` correctly extracted facts from context and cleanly handles the out-of-scope `q5` by stating missing info.
- **What could potentially go wrong / error handling**: Timeout delays due to generating responses with a larger 20B model locally. Increase timeout parameters or optimize batch execution if necessary.

---

## Section 3 — Why `assert answer == expected` fails

Run the same question three times with temperature=1.0 (max variation) and observe.

### Cell 5 Explanation

- **Objective**: Demonstrate why exact string matching (`assert candidate == reference`) fails in LLM software engineering.
- **Purpose**: Run three high-temperature generation calls on the same query using `gpt-oss:20b` to display string variance despite semantic correctness.
- **Why we run the code**: Proves empirically why model evaluation requires semantic LLM-as-a-judge approaches rather than deterministic unit tests.
- **What is industry standard**: Utilizing semantic assertion models, embeddings distance, or judge scoring rather than exact string equality check.

In [5]:
same_question = "Which parent company owns LinkedIn?"
expected_answer = "Microsoft"

print(f"Q: {same_question}")
print(f"Expected answer: {expected_answer!r}\n")

answers = []
for i in range(3):
    a = answer_naively(same_question, PLATFORMS, temperature=1.0)
    answers.append(a)
    print(f"  Attempt {i+1}: {a!r}")

print("\n═ The naive test ═")
for i, a in enumerate(answers, 1):
    equal = a == expected_answer
    print(f"  Attempt {i} == 'Microsoft'?  {equal}")

print("\nAll three answers are CORRECT. All three FAIL `assert answer == expected`.")
print("This is why we need eval — not testing.")

Q: Which parent company owns LinkedIn?
Expected answer: 'Microsoft'

  Attempt 1: 'Microsoft'
  Attempt 2: 'I’m sorry, but the answer isn’t in the provided profiles.'
  Attempt 3: 'LinkedIn is owned by **Microsoft**.'

═ The naive test ═
  Attempt 1 == 'Microsoft'?  True
  Attempt 2 == 'Microsoft'?  False
  Attempt 3 == 'Microsoft'?  False

All three answers are CORRECT. All three FAIL `assert answer == expected`.
This is why we need eval — not testing.


---
### Output Analysis — Cell 5

- **What is expected from output**: Three semantically valid variations of the answer that each result in `False` when checked against exact string equality (`== 'Microsoft'`).
- **How to understand the Output**: Confirms that non-zero temperature produces output variations in `gpt-oss:20b`, highlighting the flaw of rigid assertions.
- **What could potentially go wrong / error handling**: If temperature=1.0 produces identical strings across short runs, rerunning or using higher temperature illustrates the non-deterministic output nature.

**Discussion:**
- All three attempts got the right answer
- All three failed strict equality
- What DID work: someone reading each answer could say "yes, that mentions Microsoft and answers the question"
- That's what LLM-as-judge automates — reading + judging

The mental shift: **don't compare strings, evaluate properties.**

---

## Section 4 — Build the LLM-as-judge (rubric-based)

We build a working LLM-as-judge with structured output (Pydantic).

**The rubric** — 3 dimensions, each scored 1-4:
- **Accuracy** — is the answer factually correct given the corpus?
- **Groundedness** — does it stay within the provided context (no invention)?
- **Format** — is it well-structured and appropriately concise?

### Cell 6 Explanation

- **Objective**: Define the standard multi-dimensional evaluation rubric text prompt.
- **Purpose**: Provide explicit guidance for judging Accuracy, Groundedness, and Format on 1-4 ordinal scales.
- **Why we run the code**: Standardized criteria reduce judge variance and force model reasoning aligned with specified quality guidelines.
- **What is industry standard**: Rubrics should use clear numerical scale definitions with explicit descriptions for each numerical tier.

In [6]:
# Define the evaluation rubric prompt text string
RUBRIC = """Evaluate the answer on three dimensions. For each, give a score 1-4:

ACCURACY (1-4)
  4 = fully correct given the corpus
  3 = mostly correct, minor issues
  2 = partially correct, significant issues
  1 = incorrect or misleading

GROUNDEDNESS (1-4)
  4 = every claim traces to the corpus
  3 = mostly grounded, small unsupported additions
  2 = mix of grounded and invented content
  1 = substantially invented or hallucinated

FORMAT (1-4)
  4 = clear, appropriately concise, well-structured
  3 = clear but slightly verbose or terse
  2 = confusing structure or wrong length
  1 = badly formatted, hard to read"""

print(RUBRIC)

Evaluate the answer on three dimensions. For each, give a score 1-4:

ACCURACY (1-4)
  4 = fully correct given the corpus
  3 = mostly correct, minor issues
  2 = partially correct, significant issues
  1 = incorrect or misleading

GROUNDEDNESS (1-4)
  4 = every claim traces to the corpus
  3 = mostly grounded, small unsupported additions
  2 = mix of grounded and invented content
  1 = substantially invented or hallucinated

FORMAT (1-4)
  4 = clear, appropriately concise, well-structured
  3 = clear but slightly verbose or terse
  2 = confusing structure or wrong length
  1 = badly formatted, hard to read


---
### Output Analysis — Cell 6

- **What is expected from output**: Printed rubric string outlining the scoring dimensions.
- **How to understand the Output**: Direct confirmation of instructions that will be supplied to the judge model.
- **What could potentially go wrong / error handling**: Formatting ambiguities inside string literals can distort model comprehension; verify indentation remains clean.

### Cell 7 Explanation

- **Objective**: Formulate the structured output schema using Pydantic.
- **Purpose**: Ensure judge verdicts are validated strictly against standard boundaries (1-4 integers and non-empty explanation strings).
- **Why we run the code**: Guaranteed JSON structure guarantees programmatically parseable metrics across batch test runs.
- **What is industry standard**: Using Pydantic or JSON schema validation to enforce typed model response contracts.

In [7]:
# Structured evaluation output schema
class JudgeVerdict(BaseModel):
    accuracy: int      = Field(ge=1, le=4)
    groundedness: int  = Field(ge=1, le=4)
    format_score: int  = Field(ge=1, le=4)
    reasoning: str     = Field(min_length=20, max_length=500)

print("Schema fields:")
for name, field in JudgeVerdict.model_fields.items():
    print(f"  {name:15s}  {field.annotation.__name__}")

Schema fields:
  accuracy         int
  groundedness     int
  format_score     int
  reasoning        str


---
### Output Analysis — Cell 7

- **What is expected from output**: Schema field names and data type signatures output cleanly (`accuracy: int`, etc.).
- **How to understand the Output**: Confirms Pydantic model compilation.
- **What could potentially go wrong / error handling**: Model parsing errors occur if the backend LLM outputs scores outside bounds (e.g., 5/5 or floats). Exception handling around Pydantic validation handles non-compliant outputs.

### Cell 8 Explanation

- **Objective**: Execute the judge model (`gemma2:2b`) on a single candidate answer.
- **Purpose**: Test structured output generation using `gemma2:2b` as judge via system prompts or JSON format mode.
- **Why we run the code**: Verifies that the lightweight judge model parses rubric prompt instructions correctly and formats JSON payloads properly.
- **What is industry standard**: Using structured parsing methods or JSON format parameters to guarantee structured responses from local models.

In [8]:
def judge(question: str, answer: str, corpus: list, ideal: str = "") -> JudgeVerdict:
    """LLM-as-judge with a rubric using local gemma2:2b model as judge."""
    context = "\n\n".join(f"=== {p['title']} ===\n{p['details']}" for p in corpus)
    ideal_hint = f"\n\nIdeal answer (for reference): {ideal}" if ideal else ""
    
    system_instructions = (
        f"You are a strict evaluator of LLM answers.\n\n{RUBRIC}\n\n"
        "Return ONLY valid JSON matching this exact structure:\n"
        "{\n"
        '  "accuracy": <int 1-4>,\n'
        '  "groundedness": <int 1-4>,\n'
        '  "format_score": <int 1-4>,\n'
        '  "reasoning": "<string 20-500 chars explaining scores>"\n'
        "}"
    )

    # Standard chat completion call configured for local Ollama compatibility
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_instructions},
            {"role": "user", "content":
                f"Corpus:\n{context}\n\n"
                f"Question: {question}{ideal_hint}\n\n"
                f"Candidate answer to evaluate: {answer}"},
        ],
    )
    raw_json = resp.choices[0].message.content
    return JudgeVerdict.model_validate_json(raw_json)

# Test judge execution on first candidate
q = QUESTIONS[0]
c = candidates[0]
verdict = judge(q["question"], c["answer"], PLATFORMS, ideal=q["ideal"])

print(f"Q: {q['question']}")
print(f"A: {c['answer']}\n")
print(f"  Accuracy:     {verdict.accuracy}/4")
print(f"  Groundedness: {verdict.groundedness}/4")
print(f"  Format:       {verdict.format_score}/4")
print(f"  Reasoning:    {verdict.reasoning}")

Q: Which parent company owns LinkedIn?
A: LinkedIn is operated by **Microsoft**.

  Accuracy:     3/4
  Groundedness: 4/4
  Format:       4/4
  Reasoning:    The answer accurately identifies that LinkedIn is operated by Microsoft. The provided text from the corpus clearly states this fact. The answer is well-formatted and concise, directly answering the question. The answer is grounded in the provided corpus, as it directly quotes the text from the corpus. 


---
### Output Analysis — Cell 8

- **What is expected from output**: A formatted breakdown displaying integer scores for Accuracy, Groundedness, and Format (e.g. 4/4) accompanied by detailed reasoning text from `gemma2:2b`.
- **How to understand the Output**: Confirms structured evaluation parsing from `gemma2:2b` with Pydantic type validation.
- **What could potentially go wrong / error handling**: Smaller judge models (`gemma2:2b`) may occasionally omit JSON fields or produce formatting errors. Standard Pydantic try-except blocks handle parse errors.

### Cell 9 Explanation

- **Objective**: Run rubric evaluations across the full suite of candidates.
- **Purpose**: Batch process all 6 generated candidate responses using `gemma2:2b` as judge.
- **Why we run the code**: Aggregates metric scores across varying question difficulties to discover model evaluation performance.
- **What is industry standard**: Running automated evaluations systematically across categorized test suites and capturing diagnostic outputs.

In [9]:
# Batch run judge evaluation over all candidate outputs
verdicts = []
for q, c in zip(QUESTIONS, candidates):
    v = judge(q["question"], c["answer"], PLATFORMS, ideal=q["ideal"])
    verdicts.append({"q_id": q["id"], "difficulty": q["difficulty"], "verdict": v, "answer": c["answer"]})
    print(f"  {q['id']} ({q['difficulty']:22s}) acc={v.accuracy} ground={v.groundedness} fmt={v.format_score}")

  q1 (easy                  ) acc=3 ground=4 fmt=4
  q2 (easy                  ) acc=3 ground=4 fmt=4
  q3 (medium                ) acc=3 ground=4 fmt=3


ValidationError: 1 validation error for JudgeVerdict
reasoning
  String should have at most 500 characters [type=string_too_long, input_value="The answer accurately de...ise, and easy to read. ", input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_long

---
### Output Analysis — Cell 9

- **What is expected from output**: Six printed summary lines displaying question IDs, difficulties, and individual metric scores calculated by `gemma2:2b`.
- **How to understand the Output**: Shows score distributions across categories to evaluate candidate consistency and judge agreement.
- **What could potentially go wrong / error handling**: Timeout errors during bulk model execution. Add retry loops or optimize hardware settings for local models.

### Cell 10 Explanation

- **Objective**: Tabulate evaluation verdicts with truncated explanations.
- **Purpose**: Render a structured comparison view for executive analysis.
- **Why we run the code**: Summarizes complex multi-dimensional evaluations into a clear format for quick review.
- **What is industry standard**: Formatting eval results into structured tables showing scores alongside justification excerpts.

In [10]:
# Render Tabular Summary
print(f"  {'Q':4s} {'Difficulty':22s} {'Acc':>4s} {'Grd':>4s} {'Fmt':>4s}  Reasoning (first 60 chars)")
print(f"  {'---':4s} {'----------':22s} {'---':>4s} {'---':>4s} {'---':>4s}  {'-'*40}")
for row in verdicts:
    v = row["verdict"]
    print(f"  {row['q_id']:4s} {row['difficulty']:22s} "
          f"{v.accuracy:>4d} {v.groundedness:>4d} {v.format_score:>4d}  "
          f"{v.reasoning[:60]}...")

  Q    Difficulty              Acc  Grd  Fmt  Reasoning (first 60 chars)
  ---  ----------              ---  ---  ---  ----------------------------------------
  q1   easy                      3    4    4  The answer accurately identifies that LinkedIn is operated b...
  q2   easy                      3    4    4  The answer accurately identifies the main recommendation fee...
  q3   medium                    3    4    3  The answer accurately reflects the monetization methods of Y...


---
### Output Analysis — Cell 10

- **What is expected from output**: Alignment table showing scores for Accuracy, Groundedness, and Format alongside the first 60 characters of reasoning.
- **How to understand the Output**: Allows immediate identification of low-scoring queries and reasoning highlights provided by `gemma2:2b`.
- **What could potentially go wrong / error handling**: Text alignment issues caused by unexpected string lengths. Solved using fixed-width string formatting operators (`:22s`, `:>4d`).

**Discussion moment:**
- Look at the reasoning field for each row. Does the judge's rationale match what YOU'D say if you were grading?
- Which question got the lowest score? Why?
- What did q5 (out-of-scope Snapchat) get? Did the naive answer refuse, or invent?
- What did q6 (inference — 'which involve moodboards and product links?') get?

**What you just built** is the same pattern you'll port to `src/eval/judge.py` in the upcoming capstone lab. The Pydantic schema, the rubric-in-system-prompt, the structured output — all of it transfers directly.

---

## Section 5 — Pairwise comparison

Rubric scoring gives you numbers. Pairwise gives you a cleaner signal: 'which of these two answers is better?' Less biased than absolute scoring.

We'll write two variants of the answer generator (different system prompts) using `gpt-oss:20b` and pit them against each other using `gemma2:2b` as judge.

### Cell 11 Explanation

- **Objective**: Generate paired candidate responses using brief vs. thorough system prompt instructions with `gpt-oss:20b`.
- **Purpose**: Produce two distinct candidate strategies (Variant A: Brief vs. Variant B: Thorough) from `gpt-oss:20b` to compare directly.
- **Why we run the code**: Pairwise comparisons require distinct generation approaches to determine preference tradeoffs.
- **What is industry standard**: A/B testing prompt configurations systematically against identical input context.

In [11]:
# Two competing system prompts
SYSTEM_A_BRIEF = (
    "Answer using only the provided platform profiles. Be brief — one sentence when possible."
)

SYSTEM_B_THOROUGH = (
    "Answer using only the provided platform profiles. Provide context and explain your "
    "reasoning. Be thorough but stay grounded."
)

def answer_with_prompt(question: str, corpus: list, system_prompt: str) -> str:
    context = "\n\n".join(f"=== {p['title']} ===\n{p['details']}" for p in corpus)
    resp = client.chat.completions.create(
        model=CANDIDATE_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Profiles:\n{context}\n\nQuestion: {question}"},
        ],
    )
    return resp.choices[0].message.content

# Generate both variants for all questions using candidate model (gpt-oss:20b)
print("Generating variant A (brief)...")
answers_a = [answer_with_prompt(q["question"], PLATFORMS, SYSTEM_A_BRIEF) for q in QUESTIONS]
print("Generating variant B (thorough)...")
answers_b = [answer_with_prompt(q["question"], PLATFORMS, SYSTEM_B_THOROUGH) for q in QUESTIONS]

# Show comparison sample
print(f"\nExample — {QUESTIONS[3]['question']}\n")
print(f"A (brief):    {answers_a[3]}\n")
print(f"B (thorough): {answers_b[3]}")

Generating variant A (brief)...
Generating variant B (thorough)...

Example — Compare how engagement formats differ between X (Twitter) and Instagram.

A (brief):    X centers on short text posts, retweets, quote‑tweets, and hashtag‑driven real‑time conversation, whereas Instagram emphasizes visual media—photos, Reels, Stories, and direct messaging—alongside visual hashtag curation and influencer‑style branding.

B (thorough): **Engagement formats on X (formerly Twitter) vs. Instagram**

| Feature | X (Twitter) | Instagram |
|---------|-------------|-----------|
| **Primary content type** | Short‑form text (tweets) | Visual media (photos, videos, Reels) |
| **Post length** | 280 characters (plus media) | Unlimited length for photos/videos; Reels up to 60 s |
| **Real‑time interaction** | Core to the platform – “live” conversations, breaking news, trending topics | Less emphasis on real‑time; content is curated in a feed and “Explore” page |
| **Threading** | Users can chain tweets into

---
### Output Analysis — Cell 11

- **What is expected from output**: Side-by-side text display showing the brief concise candidate answer (A) versus the detailed response (B) generated by `gpt-oss:20b`.
- **How to understand the Output**: Verifies that system prompts produced noticeable structural variations in `gpt-oss:20b` responses.
- **What could potentially go wrong / error handling**: If `gpt-oss:20b` ignores prompt constraints (e.g. producing long answers for brief variant), refine system instructions with explicit length controls.

### Cell 12 Explanation

- **Objective**: Execute pairwise comparisons using `gemma2:2b` as judge model.
- **Purpose**: Present side-by-side responses to the judge model (`gemma2:2b`) to determine winning candidates along with reasoning.
- **Why we run the code**: Pairwise comparative evaluations provide higher sensitivity when evaluating subtle prompt modifications.
- **What is industry standard**: Enforcing constrained winner selection (A vs B) accompanied by chain-of-thought rationale.

In [12]:
# Schema for Pairwise Verdict
class PairwiseVerdict(BaseModel):
    winner: str        = Field(pattern="^[AB]$")  # 'A' or 'B'
    reasoning: str     = Field(min_length=20, max_length=400)

def pairwise_judge(question: str, answer_a: str, answer_b: str, corpus: list) -> PairwiseVerdict:
    """Show the judge two answers side-by-side using local gemma2:2b. Ask which is better."""
    context = "\n\n".join(f"=== {p['title']} ===\n{p['details']}" for p in corpus)
    system_instructions = (
        "You are comparing two candidate answers to the same question. "
        "Judge on: accuracy, groundedness, and appropriate length/format. "
        "Pick A or B as the winner.\n\n"
        "Return ONLY valid JSON with format:\n"
        '{"winner": "A" or "B", "reasoning": "<explanation 20-400 chars>"}'
    )
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_instructions},
            {"role": "user", "content":
                f"Corpus:\n{context}\n\n"
                f"Question: {question}\n\n"
                f"Answer A: {answer_a}\n\n"
                f"Answer B: {answer_b}"},
        ],
    )
    raw = resp.choices[0].message.content
    return PairwiseVerdict.model_validate_json(raw)

# Run pairwise on all 6
results = []
for q, a, b in zip(QUESTIONS, answers_a, answers_b):
    v = pairwise_judge(q["question"], a, b, PLATFORMS)
    results.append({"q_id": q["id"], "winner": v.winner, "reasoning": v.reasoning})
    print(f"  {q['id']}  winner={v.winner}   {v.reasoning[:70]}...")

wins_a = sum(1 for r in results if r["winner"] == "A")
wins_b = sum(1 for r in results if r["winner"] == "B")
print(f"\nTally: A (brief) won {wins_a}   |   B (thorough) won {wins_b}")

  q1  winner=B   Answer B is a more accurate and concise response. It directly states t...


ValidationError: 1 validation error for PairwiseVerdict
  Invalid JSON: EOF while parsing an object at line 16 column 2 [type=json_invalid, input_value='{"winner": "A", "reasoni...n  \n  \n  \n  \n  \n  ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid

---
### Output Analysis — Cell 12

- **What is expected from output**: Per-question win designations (`winner=A` or `winner=B`) followed by justification and an overall tally.
- **How to understand the Output**: Displays how `gemma2:2b` weighs brevity against thoroughness across simple facts versus multi-hop reasoning.
- **What could potentially go wrong / error handling**: If judge output fails standard parsing, default fallbacks can handle token parsing or re-prompting.

**Discussion:**
- Which prompt variant won overall?
- On which questions? (E.g., did 'thorough' win on the multi-fact q4?)
- Would you have picked the same winners?
- **But wait** — did A win because A really is better, or because A was always shown first? That's Section 6.

---

## Section 6 — Position bias, live

LLM judges tend to prefer the first option shown. This is **position bias** — we see it happen.

**Experiment:** run the same pairwise 3 times with the order flipped between runs. If the judge flips its winner just because the order changed, that's position bias caught in the act.

### Cell 13 Explanation

- **Objective**: Demonstrate position bias vulnerability in LLM judges.
- **Purpose**: Run pairwise judge calls on identical text while swapping presentation order (A/B vs B/A).
- **Why we run the code**: Evaluates whether position placement alters the judge model's decision.
- **What is industry standard**: Documenting and actively testing models for structural, position, and verbosity evaluation biases.

In [13]:
test_q  = QUESTIONS[3]["question"]  # multi-fact question
ans_a   = answers_a[3]              # brief variant
ans_b   = answers_b[3]              # thorough variant

print(f"Q: {test_q}\n")
print("Running 3 rounds. Watch if the judge flips.\n")

# Round 1: A first, B second
r1 = pairwise_judge(test_q, ans_a, ans_b, PLATFORMS)
print(f"  Round 1  (A shown first, B second):  winner={r1.winner}")
print(f"           {r1.reasoning[:80]}...\n")

# Round 2: B first, A second (flipped)
r2 = pairwise_judge(test_q, ans_b, ans_a, PLATFORMS)
print(f"  Round 2  (B shown first, A second):  winner={r2.winner}")
print(f"           NOTE: 'winner' is A or B by position, not by content.")
print(f"           So 'A' in Round 2 refers to the THOROUGH answer (originally B).\n")

# Round 3: A first again
r3 = pairwise_judge(test_q, ans_a, ans_b, PLATFORMS)
print(f"  Round 3  (A first again):            winner={r3.winner}")

print("\nDid the winner flip between rounds 1 and 2?")
print("If yes: position bias caught in the act.")
print("If no: this judge is more robust — but bias still exists on other questions.")

Q: Compare how engagement formats differ between X (Twitter) and Instagram.

Running 3 rounds. Watch if the judge flips.



ValidationError: 1 validation error for PairwiseVerdict
reasoning
  String should have at most 400 characters [type=string_too_long, input_value='Answer B provides a deta...ween the two platforms.', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_long

---
### Output Analysis — Cell 13

- **What is expected from output**: Printed outcomes across 3 evaluation rounds tracking positional choices.
- **How to understand the Output**: Shows whether winner decisions shift based on presentation order.
- **What could potentially go wrong / error handling**: If the model consistently selects position 1 regardless of content, position bias is observed and must be addressed using random swapping or debiased aggregation.

### Cell 14 Explanation

- **Objective**: Implement position-debiasing logic for pairwise evaluations.
- **Purpose**: Randomize candidate presentation order and dynamically map winner decisions back to original candidate labels.
- **Why we run the code**: Position-swapping neutralizes positional preference bias in pairwise comparisons.
- **What is industry standard**: Employing randomized position swapping or bidirectional evaluation passes (evaluating both A/B and B/A and checking consistency).

In [14]:
def pairwise_judge_debiased(question: str, answer_variant_a: str,
                            answer_variant_b: str, corpus: list,
                            rng: random.Random | None = None) -> dict:
    """Flip position randomly. Return winner in ORIGINAL A/B labels (not shown order)."""
    rng = rng or random.Random()
    flip = rng.random() < 0.5
    
    if flip:
        # Show B first, A second
        v = pairwise_judge(question, answer_variant_b, answer_variant_a, corpus)
        # 'A' in response corresponds to option shown first -> variant B
        actual_winner = "B" if v.winner == "A" else "A"
    else:
        # Show A first
        v = pairwise_judge(question, answer_variant_a, answer_variant_b, corpus)
        actual_winner = v.winner
    
    return {"winner": actual_winner, "flipped": flip, "reasoning": v.reasoning}

# Run 5 rounds with debiasing — random flip each time
rng = random.Random(42)  # seeded for reproducibility

print(f"Q: {test_q}\n")
print("5 debiased rounds (random position-flip each time):\n")

for i in range(5):
    r = pairwise_judge_debiased(test_q, ans_a, ans_b, PLATFORMS, rng=rng)
    order = "B first" if r["flipped"] else "A first"
    print(f"  Round {i+1}  ({order})  actual_winner={r['winner']}")

print("\nWith random flipping, the winner should reflect content, not position.")

Q: Compare how engagement formats differ between X (Twitter) and Instagram.

5 debiased rounds (random position-flip each time):



ValidationError: 1 validation error for PairwiseVerdict
reasoning
  String should have at most 400 characters [type=string_too_long, input_value='Answer B provides a deta...ween the two platforms.', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_long

---
### Output Analysis — Cell 14

- **What is expected from output**: Five logged rounds showing random order assignment (`A first` or `B first`) mapped consistently to the `actual_winner` variant.
- **How to understand the Output**: Confirms that evaluation results remain consistent regardless of input presentation order.
- **What could potentially go wrong / error handling**: Logical mapping errors in label inversion (`"B" if v.winner == "A" else "A"`). Comprehensive unit test coverage ensures remap correctness.

**Discussion moment:**
- Section 5 gave us a tally. Was it reliable?
- The mitigation in 6b is what real systems use. In your upcoming capstone lab, apply position-flip when you run pairwise on YOUR two prompts.
- Other judge biases exist too (length bias — judges prefer longer answers; self-bias — judges prefer their own model). We're just doing position today.

---

## Section 7 — Critic-Creator loop

The final eval pattern. Two LLMs (or two prompts) alternate:
- **Creator** (`gpt-oss:20b`) drafts an answer
- **Critic** (`gemma2:2b`) critiques it using the rubric
- **Creator** revises based on the critique
- Repeat 2-3 rounds

Watch the answer improve round-by-round.

### Cell 15 Explanation

- **Objective**: Define the Creator agent logic using `gpt-oss:20b`.
- **Purpose**: Draft initial responses and iteratively incorporate feedback into updated revisions using heavy reasoning abilities.
- **Why we run the code**: Implements the generator component of the iterative refinement loop.
- **What is industry standard**: Decoupling drafting (Creator) from evaluation (Critic) roles to implement multi-agent refinement patterns.

In [15]:
def creator(question: str, corpus: list, previous_critique: str = "") -> str:
    """Draft (or revise) an answer using candidate model (gpt-oss:20b). If a critique is provided, address it."""
    context = "\n\n".join(f"=== {p['title']} ===\n{p['details']}" for p in corpus)
    
    if previous_critique:
        user_msg = (
            f"Corpus:\n{context}\n\n"
            f"Question: {question}\n\n"
            f"Previous critique to address:\n{previous_critique}\n\n"
            f"Write an improved answer that addresses the critique."
        )
    else:
        user_msg = (
            f"Corpus:\n{context}\n\n"
            f"Question: {question}\n\n"
            f"Write a clear, grounded answer."
        )
    
    resp = client.chat.completions.create(
        model=CANDIDATE_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content":
                "You are the Creator. Draft answers using only the provided corpus."},
            {"role": "user", "content": user_msg},
        ],
    )
    return resp.choices[0].message.content

---
### Output Analysis — Cell 15

- **What is expected from output**: Silent cell execution compiling `creator` function definitions into memory.
- **How to understand the Output**: Establishes the Creator role function.
- **What could potentially go wrong / error handling**: Ensure function handles empty `previous_critique` strings without throwing formatting errors.

### Cell 16 Explanation

- **Objective**: Define the Critic agent logic using `gemma2:2b`.
- **Purpose**: Analyze drafts against rubric criteria and supply actionable feedback using the judge model.
- **Why we run the code**: Employs the judge model to identify flaws in candidate drafts.
- **What is industry standard**: Providing structured critiques with targeted modification directives rather than generic feedback.

In [16]:
def critic(question: str, draft: str, corpus: list) -> str:
    """Critique a draft answer using the judge model (gemma2:2b) and rubric."""
    context = "\n\n".join(f"=== {p['title']} ===\n{p['details']}" for p in corpus)
    
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0.0,
        messages=[
            {"role": "system", "content":
                f"You are the Critic. Read a draft answer and critique it using "
                f"this rubric:\n\n{RUBRIC}\n\n"
                "Point out specific weaknesses. Be constructive — say WHAT to "
                "change and HOW. If the draft is already strong, say so."},
            {"role": "user", "content":
                f"Corpus:\n{context}\n\n"
                f"Question: {question}\n\n"
                f"Draft answer to critique:\n{draft}"},
        ],
    )
    return resp.choices[0].message.content

---
### Output Analysis — Cell 16

- **What is expected from output**: Silent execution confirming definition of `critic` function.
- **How to understand the Output**: Prepares the Critic feedback generator for interactive execution.
- **What could potentially go wrong / error handling**: Ensure system prompts contain full rubric criteria so feedback aligns with grading parameters.

### Cell 17 Explanation

- **Objective**: Run the multi-turn Critic-Creator loop with flipped roles (`gpt-oss:20b` creator / `gemma2:2b` critic).
- **Purpose**: Execute 3 iterative rounds of generation, critique, and refinement on a complex test query.
- **Why we run the code**: Demonstrates automated quality improvement through feedback loops.
- **What is industry standard**: Setting capped iteration limits (2-3 rounds) to manage token usage and latency.

In [17]:
# Multi-turn feedback loop execution
hard_q = QUESTIONS[3]["question"]

print(f"Q: {hard_q}\n")
print("═" * 70)

# Round 1: Creator draft (gpt-oss:20b), Critic critique (gemma2:2b)
draft_1 = creator(hard_q, PLATFORMS)
print(f"\n── ROUND 1 · Creator draft ──\n{draft_1}")

critique_1 = critic(hard_q, draft_1, PLATFORMS)
print(f"\n── ROUND 1 · Critic critique ──\n{critique_1}")

# Round 2: Creator revision based on critique
draft_2 = creator(hard_q, PLATFORMS, previous_critique=critique_1)
print(f"\n── ROUND 2 · Creator revision ──\n{draft_2}")

critique_2 = critic(hard_q, draft_2, PLATFORMS)
print(f"\n── ROUND 2 · Critic critique ──\n{critique_2}")

# Round 3: Creator final pass
draft_3 = creator(hard_q, PLATFORMS, previous_critique=critique_2)
print(f"\n── ROUND 3 · Creator final ──\n{draft_3}")

Q: Compare how engagement formats differ between X (Twitter) and Instagram.

══════════════════════════════════════════════════════════════════════

── ROUND 1 · Creator draft ──
**Engagement formats on X (formerly Twitter) vs. Instagram**

| Feature | X (Twitter) | Instagram |
|---------|-------------|-----------|
| **Primary content type** | Short text posts (tweets) | Photos and short videos (Reels) |
| **Ephemeral content** | Not a core feature (no “Stories”‑style disappearing posts) | Stories – vertical, time‑limited photos or videos |
| **Short‑form video** | Not highlighted in the corpus | Reels – short, edited videos that can be discovered via the “For You”‑style feed |
| **Direct communication** | Re‑posts, direct quotes, and subscription‑based premium messaging | Direct messaging (DMs) |
| **Hashtag use** | Hashtags are used for tagging and discovery, but the focus is on text | Visual hashtag curation is a key part of discovery and feed organization |
| **Discovery & feeds** 

---
### Output Analysis — Cell 17

- **What is expected from output**: Sequential logs displaying the draft, critique, revision, second critique, and final polished response.
- **How to understand the Output**: Displays how feedback from `gemma2:2b` drives qualitative response improvements in `gpt-oss:20b` across iteration rounds.
- **What could potentially go wrong / error handling**: Diminishing returns or infinite feedback loops. Set maximum round cutoffs (`max_rounds=3`) to prevent runaway API calls.

**Discussion moment:**
- Did the answer materially improve from Round 1 to Round 3?
- What specific weaknesses did the critic catch that the creator addressed?
- Did any critique feel wrong or nit-picky?
- When would this loop stop being useful? (Usually 2-3 rounds; more rarely helps.)

**Where you'll use this pattern:**
- In the upcoming capstone lab on one of YOUR harder golden questions
- In production: some systems run Critic-Creator on ambiguous or high-stakes answers before returning them to users. Expensive but effective.

---

## Section 8 — Wrap: the 4 patterns you'll use next

In 90 minutes you built:

| Pattern | Section | What it gives you | Cost |
|---|---|---|---|
| **Rubric-based LLM-as-judge** | 4 | Scores on 3 dimensions with reasoning | 1 judge call per (q, a) |
| **Pairwise comparison** | 5 | 'Which is better?' cleaner signal than absolute scoring | 1 judge call per (q, a_A, a_B) |
| **Position-bias mitigation** | 6 | Random-flip to remove ordering bias | Same cost, cleaner result |
| **Critic-Creator loop** | 7 | Systematic prompt improvement | 2-3 rounds × 2 calls per round |

**Upcoming capstone lab:**

1. **Stakeholder Map** for YOUR capstone — new material
2. **Build 20-question golden set** for YOUR corpus — new material
3. **Port `judge` to `src/eval/judge.py`** and run on YOUR golden set — Section 4 applied
4. **Port `pairwise_judge_debiased` to `src/eval/pairwise.py`** and run between YOUR two prompts — Section 5 + 6 applied
5. **Run Critic-Creator on ONE hard golden question** — Section 7 applied
6. **Finalise ADR v1 (Locked) + DR #1 1-pager** — M1 close-out

The notebook code isn't the final answer — next you'll integrate these into `src/eval/*` modules with SQLite persistence, prompt versioning, and proper error handling. Same shape, capstone-integrated.

**That is your M1 deliverable.**